# ConFit Fine-Tuning — Notebook 2: Data Preparation

Builds the training-ready sentence-pair dataset in four steps:

| Step | What | Output |
|------|------|--------|
| 1 | Extract skill **evidence** sentences from resumes (Nvidia LLM) | `resume_evidence.jsonl` |
| 2 | Extract skill **requirement** sentences from JDs (Nvidia LLM) | `jd_requirements.jsonl` |
| 3 | Normalize all raw skill names → O*NET canonical IDs (Qdrant + Nvidia LLM, reuses `app/services/skill_normalizer`) | `skill_cache.json`, both normalized JSONLs |
| 4 | Build positive pairs + hard negatives | (NB3 input) |

**Hardware assumption:** local cluster with access to NVIDIA API + Qdrant Cloud + Neo4j Aura (same as production app).

**Datasets confirmed from NB1:**
- `Resume.csv` — 2,484 resumes, columns: `ID`, `Resume_str`, `Category` (24 UPPERCASE labels)
- `jobs.csv` — 2,277 JDs, columns: `Job Title`, `Job Description`

## Section 1 — Install Dependencies

In [ ]:
%pip install -q kagglehub openai qdrant-client sentence-transformers neo4j nest_asyncio python-dotenv pandas tqdm

## Section 2 — Imports, Paths & App Bootstrap

Adds the project root to `sys.path` and loads `.env` so the existing `app/` modules (skill_normalizer, embedding_client, vector_client, nvidia_llm_client) can be imported without modification.

In [ ]:
import sys
import os
import json
import re
import time
import asyncio
import uuid
import warnings
from pathlib import Path

import nest_asyncio
import pandas as pd
import kagglehub
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

# ── Project root resolution ──────────────────────────────────────────────────
# Notebook lives at  <root>/machine_learning/02_data_preparation.ipynb
# Project root is one level up.
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Change cwd to root so pydantic-settings can find .env
os.chdir(PROJECT_ROOT)

# ── Apply nest_asyncio so we can use asyncio.run() inside Jupyter ────────────
nest_asyncio.apply()

# ── Output directories ────────────────────────────────────────────────────────
OUT_DIR = NOTEBOOK_DIR / "data" / "nb2_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESUME_EVIDENCE_FILE  = OUT_DIR / "resume_evidence.jsonl"
JD_REQUIREMENTS_FILE  = OUT_DIR / "jd_requirements.jsonl"
SKILL_CACHE_FILE      = OUT_DIR / "skill_cache.json"
RESUME_NORM_FILE      = OUT_DIR / "resume_evidence_normalized.jsonl"
JD_NORM_FILE          = OUT_DIR / "jd_requirements_normalized.jsonl"

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs dir  : {OUT_DIR}")
print(f"sys.path[0]  : {sys.path[0]}")

In [ ]:
# ── Import app modules (requires sys.path set above) ────────────────────────
# These reuse the exact same clients as the production app:
#   - nvidia_llm_client  → openai/gpt-oss-20b via NVIDIA API (async, streaming)
#   - embedding_client   → intfloat/multilingual-e5-small (local)
#   - vector_client      → Qdrant Cloud
#   - normalize_skill    → full 2-stage pipeline (Qdrant search + LLM judge + auto-upsert)

from app.clients.nvidia_llm_client import nvidia_llm_client
from app.clients.embedding_client  import embedding_client
from app.clients.vector_client     import vector_client
from app.services.skill_normalizer import normalize_skill
from app.config import settings

print(f"NVIDIA model        : {nvidia_llm_client.MODEL}")
print(f"Embedding model     : {embedding_client.MODEL_NAME}")
print(f"Qdrant URL          : {settings.QDRANT_URL[:40]}...")
print(f"Qdrant connected    : {vector_client.test_connection()}")

## Section 3 — Download & Load Datasets

Same kagglehub downloads as NB1 — cached locally after first run.

In [ ]:
jobs_path   = kagglehub.dataset_download("kshitizregmi/jobs-and-job-description")
resume_path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

jobs_df   = pd.read_csv(list(Path(jobs_path).rglob("*.csv"))[0])
resume_df = pd.read_csv(list(Path(resume_path).rglob("*.csv"))[0])

# ── Confirmed column names from NB1 exploration ──────────────────────────────
JOBS_TITLE_COL = "Job Title"
JOBS_TEXT_COL  = "Job Description"
RES_TEXT_COL   = "Resume_str"
RES_CAT_COL    = "Category"

# Drop nulls in critical columns
jobs_df   = jobs_df.dropna(subset=[JOBS_TITLE_COL, JOBS_TEXT_COL]).reset_index(drop=True)
resume_df = resume_df.dropna(subset=[RES_TEXT_COL, RES_CAT_COL]).reset_index(drop=True)

print(f"Jobs   : {len(jobs_df):,} rows")
print(f"Resumes: {len(resume_df):,} rows")
print(f"Resume categories ({resume_df[RES_CAT_COL].nunique()}): {sorted(resume_df[RES_CAT_COL].unique().tolist())}")

## Section 4 — LLM Extraction Helpers

The existing `nvidia_llm_client` is async. We wrap it with `asyncio.run()` for use in notebook cells.
Prompts match `ml.md` specification exactly.

In [ ]:
# ── Prompts (verbatim from ml.md) ─────────────────────────────────────────

EVIDENCE_PROMPT = """\
You are a skill evidence extractor.

Given this resume, extract skill evidence sentences.
For each skill mentioned, find the sentence or phrase that best describes \
HOW the person used that skill — their actual work, not just the skill name.

Return a JSON array ONLY — no markdown, no explanation:
[
  {{
    "skill": "raw skill name as written in resume",
    "evidence": "exact sentence or phrase describing usage",
    "depth_signal": "surface|regular|deep|expert"
  }}
]

Rules:
- evidence must describe actual work done, not just list the skill
- if resume only lists the skill with no context, set evidence = null
- depth_signal is your estimate of usage depth
- extract maximum 10 skills per resume

Resume:
{resume_text}
"""

REQUIREMENT_PROMPT = """\
You are a job requirement extractor.

Given this job description, extract skill requirement sentences.
For each required skill, find the sentence or phrase that best describes \
what level of proficiency is expected.

Return a JSON array ONLY — no markdown, no explanation:
[
  {{
    "skill": "raw skill name as written in JD",
    "requirement": "exact sentence describing what is expected",
    "level": "junior|mid|senior"
  }}
]

Rules:
- requirement must describe the expected proficiency, not just list the skill
- if no context is given, set requirement = null
- level is inferred from surrounding context (years required, responsibilities)
- extract maximum 10 skills per JD

Job Description:
{jd_text}
"""

# ── Sync wrapper: run the async nvidia_llm_client in notebook context ─────────
def llm_call(prompt: str, max_tokens: int = 2048) -> str:
    """Call nvidia_llm_client.complete() synchronously via asyncio.run()."""
    return asyncio.run(nvidia_llm_client.complete(prompt, max_tokens=max_tokens))

# ── JSON extraction: strip markdown fences and parse ─────────────────────────
_JSON_ARRAY_RE = re.compile(r"\[.*\]", re.DOTALL)

def extract_json_array(raw: str) -> list:
    """Pull the first JSON array out of an LLM reply, robustly."""
    raw = raw.strip()
    # Strip ```json ... ``` fences if present
    raw = re.sub(r"```(?:json)?", "", raw).strip().strip("`")
    m = _JSON_ARRAY_RE.search(raw)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return []

print("Prompts and helpers defined.")

## Section 5 — Step 1: Extract Skill Evidence from Resumes

Processes each resume individually. Saves results incrementally to `resume_evidence.jsonl` — **resumable**: already-processed rows are skipped on re-run.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
RESUME_CALL_DELAY = 0.5   # seconds between LLM calls (rate limit cushion)
# Max words per resume sent to LLM — long resumes are truncated to keep prompts manageable
MAX_RESUME_WORDS  = 600

# ── Find already-processed resume IDs (resumability) ─────────────────────────
processed_resume_ids: set = set()
if RESUME_EVIDENCE_FILE.exists():
    with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                processed_resume_ids.add(obj["resume_id"])
            except Exception:
                pass
    print(f"Resuming: {len(processed_resume_ids)} resumes already processed.")
else:
    print("Starting fresh — no existing evidence file.")

In [ ]:
# ── Main extraction loop ─────────────────────────────────────────────────────
evidence_rows_new = 0
evidence_skipped  = 0

with open(RESUME_EVIDENCE_FILE, "a", encoding="utf-8") as out_f:
    for _, row in tqdm(resume_df.iterrows(), total=len(resume_df), desc="Resumes"):
        resume_id  = str(row["ID"])
        category   = str(row[RES_CAT_COL])
        resume_text = str(row[RES_TEXT_COL])

        # Skip already-processed
        if resume_id in processed_resume_ids:
            evidence_skipped += 1
            continue

        # Truncate very long resumes to avoid context overflow
        words = resume_text.split()
        if len(words) > MAX_RESUME_WORDS:
            resume_text = " ".join(words[:MAX_RESUME_WORDS])

        # LLM call
        try:
            raw_reply = llm_call(
                EVIDENCE_PROMPT.format(resume_text=resume_text),
                max_tokens=1024,
            )
            skills = extract_json_array(raw_reply)
        except Exception as e:
            print(f"\n[!] Resume {resume_id} failed: {e}")
            skills = []

        # Filter: drop null evidence or evidence == skill name
        valid_skills = [
            s for s in skills
            if isinstance(s, dict)
            and s.get("evidence")
            and s["evidence"].strip().lower() != s.get("skill", "").strip().lower()
        ]

        # Write one JSONL record per resume (all its skills in one line)
        record = {
            "resume_id": resume_id,
            "category":  category,
            "skills":    valid_skills,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        processed_resume_ids.add(resume_id)
        evidence_rows_new += 1
        time.sleep(RESUME_CALL_DELAY)

print(f"\nDone. New: {evidence_rows_new}  |  Skipped (already done): {evidence_skipped}")
print(f"Output: {RESUME_EVIDENCE_FILE}")

In [ ]:
# ── Inspect evidence output ───────────────────────────────────────────────────
evidence_flat = []  # one row per (resume, skill)
with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            evidence_flat.append({
                "resume_id":   rec["resume_id"],
                "category":    rec["category"],
                "skill":       s.get("skill"),
                "evidence":    s.get("evidence"),
                "depth_signal":s.get("depth_signal"),
            })

evidence_df = pd.DataFrame(evidence_flat)
print(f"Total evidence rows : {len(evidence_df):,}")
print(f"Resumes covered     : {evidence_df['resume_id'].nunique():,}")
print(f"Depth signal dist   :\n{evidence_df['depth_signal'].value_counts().to_string()}")
print()
evidence_df.sample(min(5, len(evidence_df)))

## Section 6 — Step 2: Extract Skill Requirements from JDs

Same pattern as Step 1. Saves to `jd_requirements.jsonl`, resumable.

In [ ]:
JD_CALL_DELAY = 0.5
MAX_JD_WORDS  = 500

# ── Resumability ──────────────────────────────────────────────────────────────
processed_jd_indices: set = set()
if JD_REQUIREMENTS_FILE.exists():
    with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                processed_jd_indices.add(obj["jd_index"])
            except Exception:
                pass
    print(f"Resuming: {len(processed_jd_indices)} JDs already processed.")
else:
    print("Starting fresh — no existing JD requirements file.")

In [ ]:
jd_rows_new = 0
jd_skipped   = 0

with open(JD_REQUIREMENTS_FILE, "a", encoding="utf-8") as out_f:
    for idx, row in tqdm(jobs_df.iterrows(), total=len(jobs_df), desc="JDs"):
        jd_index  = int(idx)
        job_title = str(row[JOBS_TITLE_COL])
        jd_text   = str(row[JOBS_TEXT_COL])

        if jd_index in processed_jd_indices:
            jd_skipped += 1
            continue

        # Truncate
        words = jd_text.split()
        if len(words) > MAX_JD_WORDS:
            jd_text = " ".join(words[:MAX_JD_WORDS])

        try:
            raw_reply = llm_call(
                REQUIREMENT_PROMPT.format(jd_text=jd_text),
                max_tokens=1024,
            )
            skills = extract_json_array(raw_reply)
        except Exception as e:
            print(f"\n[!] JD {jd_index} ({job_title}) failed: {e}")
            skills = []

        # Filter: drop null requirements
        valid_skills = [
            s for s in skills
            if isinstance(s, dict) and s.get("requirement")
            and s["requirement"].strip().lower() != s.get("skill", "").strip().lower()
        ]

        record = {
            "jd_index":  jd_index,
            "job_title": job_title,
            "skills":    valid_skills,
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        processed_jd_indices.add(jd_index)
        jd_rows_new += 1
        time.sleep(JD_CALL_DELAY)

print(f"\nDone. New: {jd_rows_new}  |  Skipped (already done): {jd_skipped}")
print(f"Output: {JD_REQUIREMENTS_FILE}")

In [ ]:
# ── Inspect JD requirements output ───────────────────────────────────────────
jd_flat = []
with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            jd_flat.append({
                "jd_index":   rec["jd_index"],
                "job_title":  rec["job_title"],
                "skill":      s.get("skill"),
                "requirement":s.get("requirement"),
                "level":      s.get("level"),
            })

jd_df = pd.DataFrame(jd_flat)
print(f"Total requirement rows : {len(jd_df):,}")
print(f"JDs covered            : {jd_df['jd_index'].nunique():,}")
print(f"Level distribution     :\n{jd_df['level'].value_counts().to_string()}")
print()
jd_df.sample(min(5, len(jd_df)))

## Section 7 — Step 3: Normalize Skills via O*NET (Qdrant + Nvidia LLM)

Reuses `app.services.skill_normalizer.normalize_skill` — the **exact same pipeline** the production app uses:

1. Embed raw skill name → query Qdrant `onet_skills` top-3
2. Nvidia LLM judge picks the best match (or says NONE)
3. If NONE → LLM coins a new canonical name → auto-upserted to **both Qdrant and Neo4j**

A shared `skill_cache.json` means each unique raw skill string is only normalized once across both datasets.

In [ ]:
# ── Collect all unique raw skill names from both datasets ─────────────────────
all_raw_skills: set = set()

# From resume evidence
with open(RESUME_EVIDENCE_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            raw = s.get("skill", "").strip()
            if raw:
                all_raw_skills.add(raw)

# From JD requirements
with open(JD_REQUIREMENTS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        for s in rec.get("skills", []):
            raw = s.get("skill", "").strip()
            if raw:
                all_raw_skills.add(raw)

# Load existing cache to skip already-normalized skills
skill_cache: dict = {}
if SKILL_CACHE_FILE.exists():
    with open(SKILL_CACHE_FILE, "r", encoding="utf-8") as f:
        skill_cache = json.load(f)
    print(f"Cache loaded: {len(skill_cache)} skills already normalized.")

to_normalize = [s for s in sorted(all_raw_skills) if s not in skill_cache]
print(f"Total unique raw skills: {len(all_raw_skills):,}")
print(f"Still to normalize     : {len(to_normalize):,}")

In [ ]:
# ── Normalize loop — uses app.services.skill_normalizer exactly as in production ──
# normalize_skill is async; asyncio.run() works here because nest_asyncio is applied above.

NORM_SAVE_EVERY = 50   # flush skill_cache to disk every N skills

for i, raw_skill in enumerate(tqdm(to_normalize, desc="Normalizing skills")):
    try:
        result = asyncio.run(normalize_skill(raw_skill))
        skill_cache[raw_skill] = {
            "matched_name":  result.get("matched_name", raw_skill),
            "canonical_id":  result.get("canonical_id"),
            "onet_level":    result.get("onet_level"),
            "source":        result.get("source", "no_match"),
        }
    except Exception as e:
        print(f"\n[!] Normalization failed for '{raw_skill}': {e}")
        skill_cache[raw_skill] = {
            "matched_name":  raw_skill,
            "canonical_id":  None,
            "onet_level":    None,
            "source":        "error",
        }

    # Periodic save so progress survives interruptions
    if (i + 1) % NORM_SAVE_EVERY == 0:
        with open(SKILL_CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(skill_cache, f, indent=2)

# Final save
with open(SKILL_CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(skill_cache, f, indent=2)

print(f"\nNormalization complete. Cache size: {len(skill_cache):,} skills.")
print(f"Saved to: {SKILL_CACHE_FILE}")

# Quick source breakdown
sources = {}
for v in skill_cache.values():
    src = v.get("source", "unknown")
    sources[src] = sources.get(src, 0) + 1
print(f"\nSource breakdown: {sources}")

In [ ]:
# ── Apply normalization cache → produce normalized JSONLs ─────────────────────

def apply_norm_to_file(in_path: Path, out_path: Path, skill_key: str, text_key: str):
    """
    Read raw JSONL, apply skill_cache to each skill entry,
    and write enriched JSONL.
    
    skill_key  : key holding the raw skill name  ('skill')
    text_key   : key holding the evidence/requirement text  ('evidence' or 'requirement')
    """
    total_records = 0
    total_skills  = 0
    with open(in_path, "r", encoding="utf-8") as fin, \
         open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            rec = json.loads(line)
            enriched_skills = []
            for s in rec.get("skills", []):
                raw = s.get(skill_key, "").strip()
                norm = skill_cache.get(raw, {})
                enriched_skills.append({
                    **s,
                    "raw_skill":    raw,
                    "matched_name": norm.get("matched_name", raw),
                    "canonical_id": norm.get("canonical_id"),
                    "onet_level":   norm.get("onet_level"),
                    "norm_source":  norm.get("source", "no_match"),
                    # Only keep entries that have actual text
                    "_keep":        bool(s.get(text_key)),
                })
            rec["skills"] = [s for s in enriched_skills if s.pop("_keep", False)]
            fout.write(json.dumps(rec) + "\n")
            total_records += 1
            total_skills  += len(rec["skills"])
    return total_records, total_skills


res_records, res_skills = apply_norm_to_file(
    RESUME_EVIDENCE_FILE, RESUME_NORM_FILE,
    skill_key="skill", text_key="evidence",
)
jd_records, jd_skills = apply_norm_to_file(
    JD_REQUIREMENTS_FILE, JD_NORM_FILE,
    skill_key="skill", text_key="requirement",
)

print(f"Resume normalized: {res_records:,} records, {res_skills:,} valid evidence rows")
print(f"JD normalized    : {jd_records:,} records, {jd_skills:,} valid requirement rows")
print(f"\nOutputs:")
print(f"  {RESUME_NORM_FILE}")
print(f"  {JD_NORM_FILE}")